In [1]:
!pip install optuna

In [2]:
import optuna 
import torch
from train_generalized_earlystopping import train, bce_loss, dice_loss, bce_dice_loss, focal_loss, tversky_loss
from data import load_mri_dataframe, get_dataloaders
from BaselineUNet import BaselineUNet

In [3]:
def objective(trial):

    # 1. Define hyperparameters to be optimized
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    loss_fn_name = trial.suggest_categorical(
        "loss_fn", ["bce_loss", "dice_loss", "bce_dice_loss", "focal_loss", "tversky_loss"]
    ) # Needs to be list of strings, not functions
    lr_step_size = trial.suggest_int("step_size", 3, 7)  # for lr scheduler
    lr_gamma = trial.suggest_float("gamma", 0.1, 0.5)  # for lr scheduler

    # 2. Load data
    df = load_mri_dataframe()
    train_loader, val_loader = get_dataloaders(df, batch_size=batch_size, omit_empty_masks=True)

    # 3. Init model, optimizer, loss function
    model = BaselineUNet()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    if optimizer_name == "Adam":
        optimizer_class = torch.optim.Adam
    else:
        optimizer_class = torch.optim.SGD

    loss_fn_dict = {
        "bce_loss": bce_loss,
        "dice_loss": dice_loss,
        "bce_dice_loss": bce_dice_loss,
        "focal_loss": focal_loss,
        "tversky_loss": tversky_loss,
    }
    loss_fn = loss_fn_dict[loss_fn_name]

    # 4. Train model
    trained_model, results = train(
        model,
        train_loader,
        val_loader,
        device,
        lr=lr,
        optimizer_class=optimizer_class,
        loss_fn=loss_fn,
        epochs=30,
        lr_sched_cls=torch.optim.lr_scheduler.StepLR,
        lr_sched_kwargs={"step_size": lr_step_size, "gamma": lr_gamma},
    )

    # 5. Evaluate
    val_dice = results["history"]["val_dice"][-1]  # Last epoch dice score
    return val_dice

In [4]:
study = optuna.create_study(direction="maximize")  # Trial goal: maximize Dice score //TODO: combine with PatientPruner
study.optimize(objective, n_trials=5)  # No. of trials to run                       //reason: optune will penalize trial if stopped early

trial = study.best_trial

print("\nBest Validation Dice Score: {}".format(trial.value))

print("\nWith Parameters:")
for key, value in trial.params.items():
    print("   {}: {}".format(key, value))

[I 2025-06-06 09:28:25,273] A new study created in memory with name: no-name-8badb09d-d86d-41dd-884e-86d5100ea310


[Data] Train images: 1093 ; Val images: 280


Epoch 1/30 - Avg Train Loss: 0.6275


Epoch 2/30 - Avg Train Loss: 0.5873


Epoch 3/30 - Avg Train Loss: 0.5536


...

Epoch 19/30 - Avg Train Loss: 0.4141


Epoch 20/30 - Avg Train Loss: 0.4156


Epoch 21/30 - Avg Train Loss: 0.4108


[I 2025-06-06 09:50:28,182] Trial 4 finished with value: 0.4809785434177944 and parameters: {'lr': 2.8870020406381103e-05, 'batch_size': 8, 'optimizer': 'Adam', 'loss_fn': 'tversky_loss', 'step_size': 6, 'gamma': 0.2680292828011849}. Best is trial 4 with value: 0.4809785434177944.


Stopped early at epoch 21!

Best Validation Dice Score: 0.4809785434177944

With Parameters:
   lr: 2.8870020406381103e-05
   batch_size: 8
   optimizer: Adam
   loss_fn: tversky_loss
   step_size: 6
   gamma: 0.2680292828011849
